In [6]:
"""
Hourly Data Assimilation & Spatial Interpolation using Kriging
- Adds dynamic lapse rate via `estimate_lapse_rate()` per-hour using station data
- Elevation handled through detrend (to reference elevation) + retrend (per grid cell)
- Ordinary Kriging with auto variogram fitting (sampled for efficiency) and safe fallbacks

Pipeline layout (aligned with IDW script):
1) CONFIG
2) UTIL: time helpers, DEM loading, projection helpers
3) DATA: load hourly parquets (stations, IMERG, MRoS) and AOI filter
4) LAPSE & DEM utilities: dynamic lapse estimator, fill missing station elev
5) INTERP: kriging (detrend/retrend with per-hour lapse for temps)
6) HOURLY LOOP: iterate hours x variables, build xarray Dataset
7) SAVE: CF-compliant NetCDF
8) QUICKLOOK: simple static PNGs per sampled hour (optional)

Outputs:
- CF-compliant NetCDF with hourly predictor stacks on the DEM grid
- Quicklook PNG maps per sampled hour with stations & MRoS markers overlayed

Notes:
- Variables: temp_air, temp_dew, temp_wet, rh (stations); mros_plp_proxy (MRoS); plp (IMERG)
- Dynamic/variable-specific min_points supported (e.g., PLP/MRoS allow lower threshold)
"""

# ============================ IMPORTS ============================
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, box
import rasterio as rio
from rasterio.warp import transform_bounds, calculate_default_transform, reproject, Resampling
from rasterio.transform import xy as rio_xy, rowcol as rio_rowcol
import xarray as xr
import rioxarray # noqa: F401
from pyproj import CRS, Transformer
from scipy.spatial import cKDTree
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
from tqdm import tqdm


# Kriging
import pykrige.kriging_tools as kt
from pykrige.ok import OrdinaryKriging

In [ ]:
BASE_DIR = Path().resolve().parent
print("BASE_DIR:", BASE_DIR)

CONFIG = {
    # Time windows
    "wy_start":  "2024-10-01T00:00:00Z",
    "wy_end":    "2025-05-31T23:59:59Z",
    "test_start": "2025-03-30T00:00:00Z",   # narrow test window first
    "test_end":   "2025-04-02T23:00:00Z",
    # "test_start": "2024-10-01T00:00:00Z",   # Entire window
    # "test_end":   "2025-05-31T23:59:59Z",

    # Paths
    "dem_path":  BASE_DIR / "DEM_1km_clipped_v2_rpj.tif",   # ensure projected (meters)
    "out_dir":   BASE_DIR / "outputs/hourly_pipeline",

    # Projection fallback if DEM CRS is geographic
    "proj_fallback": "EPSG:26911",  # UTM 11N

    # Data inputs (hourly parquets produced upstream)
    "stations_parquet": BASE_DIR / "outputs/hourly_pipeline/hourly_data/stations_hourly.parquet",
    "imerg_parquet":    BASE_DIR / "outputs/hourly_pipeline/hourly_data/imerg_hourly.parquet",
    "mros_parquet":     BASE_DIR / "outputs/hourly_pipeline/hourly_data/mros_hourly.parquet",

    # Variables and lapse usage
    "variables": [
        ("temp_air",       "station", True),
        ("temp_dew",       "station", True),
        ("temp_wet",       "station", True),
        ("rh",             "station", False),
        ("mros_plp_proxy", "mros",    False),
        ("plp",            "imerg",   False),
    ],
    "min_points": {  # per-variable minimum points
        "temp_air": 4, "temp_dew": 4, "temp_wet": 4, "rh": 4,
        "mros_plp_proxy": 2, "plp": 1
    },

    # Lapse rate
    "default_lapse_degC_per_m": -0.005,     # fallback if regression fails
    "min_points_lapse": 5,                  # min stations to estimate dynamic lapse
    "lapse_bounds": (-0.009, 0.002),        # reasonable bounds (degC per m)

    # Kriging/variogram
    "variogram_model": "spherical",        # keep spherical as default
    "kriging_chunk_size": 2000,             # predict grid in chunks; how many grid points get kriged per iteration
    # Choose one strategy below
    "variogram_strategy": "auto",          # "auto" | "fixed" | "search"
    # If fixed, supply params (PyKrige accepts list [sill, range, nugget] or dict)
    "variogram_fixed_params": None,         # e.g., [1.0, 30000.0, 0.1]

    # Lightweight CV/grid-search for (sill, range, nugget)
    "variogram_search": {
        "enable": True,          # only used if strategy == "search"
        "max_points": 100,       # sample points for fitting/validation
        "kfold": 5,              # K-fold CV on points (random split)
        # modest grids; tune as needed
        "sill":   [0.5, 1.0, 2.0],
        "range":  [15000.0, 30000.0, 60000.0],
        "nugget": [0.0, 0.05, 0.1],
        "random_seed": 42
    },
}

OUT_DIR = Path(CONFIG["out_dir"]); OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_DIR: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype


In [3]:
# ============================ UTILITIES ============================
def to_utc(dt_series: pd.Series) -> pd.DatetimeIndex:
    """Force timestamps to UTC, making naive → UTC-naive assumed in UTC."""
    dt = pd.to_datetime(dt_series, errors="coerce", utc=True)
    # If dt_series had naive datetimes and pandas assumed local, .tz_convert('UTC') not needed.
    return dt

def hourly_index(start_iso: str, end_iso: str) -> pd.DatetimeIndex:
    return pd.date_range(start=pd.to_datetime(start_iso), end=pd.to_datetime(end_iso),
                         freq="H", tz="UTC")

def print_time(ts):
    return pd.to_datetime(ts).strftime("%Y-%m-%d %H:%MZ")

In [11]:
# --------------------- Load DEM ------------------------

def grid_centers(profile):
    T = profile["transform"]
    xs = T.c + (np.arange(profile["width"]) + 0.5) * T.a
    ys = T.f + (np.arange(profile["height"]) + 0.5) * T.e
    X, Y = np.meshgrid(xs, ys)
    return np.column_stack([X.ravel(), Y.ravel()])

with rio.open(CONFIG["dem_path"]) as src:
    dem_crs = src.crs
    if not dem_crs or not dem_crs.is_projected:
        print(f"DEM is geographic ({dem_crs}); reprojecting to {CONFIG['proj_fallback']} ...")
        dst_crs = CONFIG["proj_fallback"]
        transform, width, height = calculate_default_transform(src.crs, dst_crs, src.width, src.height, *src.bounds)
        kwargs = src.meta.copy(); kwargs.update({"crs": dst_crs, "transform": transform, "width": width, "height": height})
        dem_data = np.empty((height, width), dtype=np.float32)
        reproject(
            source=rio.band(src, 1), destination=dem_data,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=transform, dst_crs=dst_crs,
            resampling=Resampling.bilinear,
        )
        dem_profile = kwargs
        proj_crs = CRS.from_user_input(dst_crs)
    else:
        dem_profile = src.profile
        dem_data = src.read(1)
        proj_crs = dem_crs

grid_xy = grid_centers(dem_profile)
grid_elev = dem_data.ravel()
H, W = dem_profile["height"], dem_profile["width"]
T = dem_profile["transform"]
cols = np.arange(W); rows = np.arange(H)
x_centers = np.array([rio_xy(T, 0, c, offset="center")[0] for c in cols])
y_centers = np.array([rio_xy(T, r, 0, offset="center")[1] for r in rows])
print(f"DEM CRS: {proj_crs}, pixel ~{abs(T.a):.2f} m | grid {W} x {H}")

print(f"DEM CRS: {proj_crs}, pixel size: {abs(dem_profile['transform'].a):.2f} m")

def load_dem_and_aoi(dem_path: str):
    with rio.open(dem_path) as src:
        dem_crs = CRS.from_wkt(src.crs.to_wkt()) if src.crs else None
        bounds = src.bounds
        aoi_wgs84 = transform_bounds(src.crs, "EPSG:4326",
                                     bounds.left, bounds.bottom, bounds.right, bounds.top,
                                     densify_pts=21)
    aoi_poly = box(aoi_wgs84[0], aoi_wgs84[1], aoi_wgs84[2], aoi_wgs84[3])
    return dem_path, dem_crs, aoi_poly

dem_path, dem_crs, aoi_poly = load_dem_and_aoi(CONFIG["dem_path"])

DEM is geographic (EPSG:4326); reprojecting to EPSG:26911 ...
DEM CRS: EPSG:26911, pixel ~961.82 m | grid 146 x 260
DEM CRS: EPSG:26911, pixel size: 961.82 m


In [12]:
# ============================ DATA LOADING ============================

# Load hourly parquets (already generated upstream)
st_hr   = pd.read_parquet(CONFIG["stations_parquet"])
imerg_hr = pd.read_parquet(CONFIG["imerg_parquet"])
mros_hr  = pd.read_parquet(CONFIG["mros_parquet"])

# Time to UTC and filter window
for df, time_col in [(st_hr, "hour_utc"), (imerg_hr, "hour_utc"), (mros_hr, "hour_utc")]:
    df[time_col] = pd.to_datetime(df[time_col], utc=True, errors="coerce").dt.floor("h")

HOURS = hourly_index(CONFIG["test_start"], CONFIG["test_end"])  # inclusive hourly range

# Filter to AOI bbox in lon/lat

def filter_points_to_aoi(df: pd.DataFrame, aoi_poly) -> pd.DataFrame:
    g = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326")
    poly = gpd.GeoSeries([aoi_poly], crs="EPSG:4326").iloc[0]
    mask = g.intersects(poly)
    return df.loc[mask.values].drop(columns=["geometry"], errors="ignore")

st_hr   = filter_points_to_aoi(st_hr, aoi_poly)
imerg_hr= filter_points_to_aoi(imerg_hr, aoi_poly)
mros_hr    = filter_points_to_aoi(mros_hr, aoi_poly)

print(len(st_hr), len(imerg_hr), len(mros_hr))


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\2193629452.py:9: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  return pd.date_range(start=pd.to_datetime(start_iso), end=pd.to_datetime(end_iso),


313521 1702800 7367


In [13]:
# ============================= 4) LAPSE & DEM utils =============================

def estimate_lapse_rate(st_df: pd.DataFrame,
                        temp_col: str = "temp_air",
                        elev_col: str = "elev",
                        default_lapse: float = -0.005,
                        min_points: int = 5,
                        bounds: tuple = (-0.009, 0.002)) -> float:
    """Dynamically estimate lapse (degC/m) via OLS on temp ~ elev, bounded."""
    use = st_df.dropna(subset=[temp_col, elev_col])
    if len(use) < min_points:
        return default_lapse
    X = use[[elev_col]].values.astype(float); y = use[temp_col].values.astype(float)
    try:
        slope = LinearRegression().fit(X, y).coef_[0]
        return slope if (bounds[0] <= slope <= bounds[1]) else default_lapse
    except Exception:
        return default_lapse


def add_dem_elev_if_missing(st_df: pd.DataFrame, profile, proj_crs) -> pd.DataFrame:
    """Fill missing station elevations by nearest-neighbor sampling of DEM."""
    if "elev" not in st_df.columns:
        st_df = st_df.copy(); st_df["elev"] = np.nan
    need = st_df["elev"].isna()
    if not need.any():
        return st_df

    tf = Transformer.from_crs("EPSG:4326", proj_crs, always_xy=True)
    xx, yy = tf.transform(st_df.loc[need, "lon"].values, st_df.loc[need, "lat"].values)
    rr, cc = rio_rowcol(profile["transform"], xx, yy, op=round)
    rr = np.clip(rr, 0, profile["height"] - 1)
    cc = np.clip(cc, 0, profile["width"]  - 1)
    st_df = st_df.copy(); st_df.loc[need, "elev"] = dem_data[rr, cc]
    return st_df

In [ ]:
# ============================= 5) INTERPOLATION =============================

def _project_lonlat_to_xy(lon, lat, dst_crs):
    tf = Transformer.from_crs("EPSG:4326", dst_crs, always_xy=True)
    return tf.transform(lon, lat)


def _ok_predict_points(px, py, values, grid_xy, variogram_model,
                       variogram_params=None, chunk_size=2000):
    """Ordinary kriging predictions at point locations using PyKrige; chunked for memory."""

    # Basic stats for debugging
    print("\n--- Kriging Debug ---")
    print(f"  n points = {len(values)}")
    print(f"  value min/max = {np.nanmin(values):.3f}/{np.nanmax(values):.3f}, "
          f"mean={np.nanmean(values):.3f}, var={np.nanvar(values):.6f}")
    print(f"  variogram_model = {variogram_model}")
    print(f"  variogram_params input = {variogram_params}")

    # Validation helper
    def valid_params(vp):
        if vp is None:
            return False
        if isinstance(vp, (list, tuple)) and len(vp) == 3:
            sill, rng, nug = vp
            ok = all(np.isfinite([sill, rng, nug])) and sill > 0 and rng > 0 and nug >= 0
            if not ok:
                print(f"Invalid variogram params (sill={sill}, range={rng}, nugget={nug})")
            return ok
        return True

    if not valid_params(variogram_params):
        print("  Falling back to auto-fit (variogram_params invalid or None).")
        variogram_params = None

    try:
        OK = OrdinaryKriging(
            px, py, values,
            variogram_model=variogram_model,
            variogram_parameters=variogram_params,
            verbose=True,  # show PyKrige optimizer output
            enable_plotting=False,
            coordinates_type="euclidean"
        )
    except ValueError as e:
        print(f"PyKrige initialization error: {e}")
        print("  Re-trying with safe defaults [1.0, 30000.0, 0.1]")
        # fallback safe defaults
        OK = OrdinaryKriging(
            px, py, values,
            variogram_model=variogram_model,
            variogram_parameters=[1.0, 30000.0, 0.1],
            verbose=True,
            enable_plotting=False,
            coordinates_type="euclidean"
        )
    except Exception as e:
        print(f"Unexpected kriging setup error: {e}")
        raise

    print("  Variogram model parameters used by PyKrige:",
          getattr(OK, "variogram_model_parameters",
                  getattr(OK, "variogram_parameters", "unknown")))

    n = len(grid_xy)
    z_pred = np.full(n, np.nan, dtype=np.float32)
    for s in range(0, n, chunk_size):
        e = min(s + chunk_size, n)
        try:
            chunk_z, _ = OK.execute("points", grid_xy[s:e, 0], grid_xy[s:e, 1])
            z_pred[s:e] = np.asarray(chunk_z, dtype=np.float32)
        except Exception as e:
            print(f"Chunk {s}:{e} failed, filling NaNs.")
            z_pred[s:e] = np.nan

    print("--- End Kriging Debug ---\n")
    return z_pred


def _cv_rmse_for_params(px, py, values, params, kfold=5, seed=42, model="spherical"):
    """Simple K-fold cross validations RMSE for a given (sill, range, nugget)."""
    rng = np.random.RandomState(seed)
    n = len(values)
    idx = np.arange(n)
    rng.shuffle(idx)
    folds = np.array_split(idx, kfold)
    errs = []
    for k in range(kfold):
        val_idx = folds[k]
        tr_idx = np.setdiff1d(idx, val_idx)
        try:
            OK = OrdinaryKriging(px[tr_idx], py[tr_idx], values[tr_idx],
                                 variogram_model=model,
                                 variogram_parameters=params,
                                 verbose=False, enable_plotting=False,
                                 coordinates_type='euclidean')
            zv, _ = OK.execute('points', px[val_idx], py[val_idx])
            err = (np.asarray(zv) - values[val_idx])
            errs.append(np.nanmean(err**2))
        except Exception:
            return np.inf
    return float(np.sqrt(np.nanmean(errs))) if errs else np.inf


def _select_variogram_params(px, py, values, strategy: str, cfg):
    """Choose variogram parameters via strategy: auto | fixed | search."""
    model = CONFIG["variogram_model"]
    if strategy == "fixed":
        return cfg.get("variogram_fixed_params", None)

    if strategy == "auto":
        return None  # PyKrige auto-fit inside _ok_predict_points

    # search
    search = CONFIG["variogram_search"]
    if not search.get("enable", True):
        return None

    # sample points for speed (variogram fitting ~ O(n^2))
    nmax = int(search.get("max_points", 100))
    rng = np.random.RandomState(search.get("random_seed", 42))
    if len(values) > nmax:
        take = rng.choice(len(values), size=nmax, replace=False)
        px_s, py_s, val_s = px[take], py[take], values[take]
    else:
        px_s, py_s, val_s = px, py, values

    best_rmse, best = np.inf, None
    for sill in search.get("sill", [1.0]):
        for rge in search.get("range", [30000.0]):
            for nug in search.get("nugget", [0.0]):
                params = [sill, rge, nug]
                rmse = _cv_rmse_for_params(px_s, py_s, val_s, params,
                                           kfold=int(search.get("kfold", 5)),
                                           seed=int(search.get("random_seed", 42)),
                                           model=model)
                if rmse < best_rmse:
                    best_rmse, best = rmse, params
    if np.isfinite(best_rmse):
        print(f"    Variogram search → best RMSE={best_rmse:.3f} with params={best}")
    else:
        print("    Variogram search failed; falling back to auto-fit")
        best = None
    return best


def krige_with_lapse(hour_points: pd.DataFrame, grid_xy: np.ndarray, grid_elev: np.ndarray,
                      proj_crs, value_col: str, station_elev_col: str,
                      apply_lapse: bool, lapse_degC_per_m: float,
                      min_points: int) -> np.ndarray:
    """Detrend to ref elevation (mean grid elev) if apply_lapse, krige, then retrend to cell elev."""
    pts = hour_points.dropna(subset=[value_col, "lon", "lat"]).copy()
    if pts.empty or pts[value_col].notna().sum() < min_points:
        return np.full(grid_elev.shape, np.nan, dtype=np.float32)

    # Ensure station elevations
    if station_elev_col not in pts.columns:
        pts[station_elev_col] = 0.0

    # Project to DEM CRS
    px, py = _project_lonlat_to_xy(pts["lon"].values, pts["lat"].values, proj_crs)
    vals = pts[value_col].values.astype(float)

    # Detrend to reference elevation
    if apply_lapse:
        stn_z = pts[station_elev_col].values.astype(float)
        ref_elev = float(np.nanmean(grid_elev)) if np.isfinite(grid_elev).any() else float(np.nanmean(stn_z))
        if not np.isfinite(ref_elev):
            ref_elev = 0.0
        vals = vals + lapse_degC_per_m * (ref_elev - stn_z)
    else:
        ref_elev = 0.0

    # Choose variogram params
    vparams = _select_variogram_params(px, py, vals, CONFIG["variogram_strategy"], CONFIG)

    # Kriging on detrended values
    z_det = _ok_predict_points(px, py, vals, grid_xy,
                               variogram_model=CONFIG["variogram_model"],
                               variogram_params=vparams,
                               chunk_size=int(CONFIG["kriging_chunk_size"]))

    # Retrend to each grid cell elevation
    if apply_lapse:
        z = z_det + lapse_degC_per_m * (grid_elev - ref_elev)
    else:
        z = z_det

    return z.astype(np.float32)

In [23]:
# ============================= 6) HOURLY LOOP =============================

coords = {"time": HOURS, "y": y_centers, "x": x_centers}
var_names = [v[0] for v in CONFIG["variables"]]
data_vars = {name: np.full((len(HOURS), H, W), np.nan, dtype=np.float32) for name in var_names}

for ti, t in enumerate(tqdm(HOURS, desc="Hourly surfaces", ncols=88)):
    st_t   = st_hr[st_hr["hour_utc"] == t]
    imerg_t= imerg_hr[imerg_hr["hour_utc"] == t]
    mros_t = mros_hr[mros_hr["hour_utc"] == t]

    # Ensure station elevs present for lapse
    st_t = add_dem_elev_if_missing(st_t, dem_profile, proj_crs)

    # Dynamic lapse from temp_air
    lapse_now = estimate_lapse_rate(
        st_t, temp_col="temp_air",
        default_lapse=CONFIG["default_lapse_degC_per_m"],
        min_points=CONFIG["min_points_lapse"],
        bounds=CONFIG["lapse_bounds"],
    )
    print(f"[{print_time(t)}] dynamic lapse = {lapse_now:.4f} °C/m")

    for name, src, use_lapse in CONFIG["variables"]:
        min_pts = CONFIG["min_points"].get(name, 3)

        if src == "station":
            if name not in st_t.columns:
                continue
            pts = st_t[["lon", "lat", "elev", name]].dropna(subset=[name])
        elif src == "imerg":
            pts = imerg_t.rename(columns={"plp": name})[["lon", "lat", name]].assign(elev=0.0)
        elif src == "mros":
            pts = mros_t.rename(columns={"mros_plp_proxy": name})[["lon", "lat", name]].assign(elev=0.0)
        else:
            continue

        if pts[name].notna().sum() < min_pts:
            print(f"    {name}: insufficient points ({pts[name].notna().sum()} < {min_pts})")
            continue
        
        # HANDLING FOR MRoS PROXY (discrete / constant cases): fractionalize and introduce small random noise to preserve ordinal meaning but allow nonzero semivariance
        if name == "mros_plp_proxy":
            pts[name] = pts[name]/100.0 + np.random.uniform(-0.02, 0.02, len(pts))
            print(f"MRoS proxy variance @ {t}: {pts[name].var():.4f}")


        lapse_apply = float(lapse_now) if use_lapse else 0.0
        try:
            vals = krige_with_lapse(
                hour_points=pts, grid_xy=grid_xy, grid_elev=grid_elev, proj_crs=proj_crs,
                value_col=name, station_elev_col="elev", apply_lapse=use_lapse,
                lapse_degC_per_m=lapse_apply, min_points=min_pts,
            )
            print(f"[{print_time(t)}] → Variable: {name}, Source: {src}, "
                f"points={len(pts)}, lapse_apply={lapse_apply:.5f}")
            if len(pts):
                print(f"    min={pts[name].min():.3f}, max={pts[name].max():.3f}, "
                    f"mean={pts[name].mean():.3f}, var={pts[name].var():.6f}")
            if name == "mros_plp_proxy":
                vals = np.clip(vals * 100.0, 0.0, 100.0)

        except Exception as e:
            print(f"Kriging failed for {name} @ {t}: {e}")
            vals = np.full(grid_elev.shape, np.nan)
        
        data_vars[name][ti, :, :] = vals.reshape(H, W)

Hourly surfaces:   0%|                                           | 0/96 [00:00<?, ?it/s]

[2025-03-30 00:00Z] dynamic lapse = -0.0037 °C/m

--- Kriging Debug ---
  n points = 40
  value min/max = -0.364/19.760, mean=6.813, var=18.282204
  variogram_model = spherical
  variogram_params input = None
  Falling back to auto-fit (variogram_params invalid or None).
Adjusting data for anisotropy...
Initializing variogram model...
Coordinates type: 'euclidean' 

Using 'spherical' Variogram Model
Partial Sill: 8.825336820390604
Full Sill: 14.891264665389347
Range: 11125.217724315955
Nugget: 6.065927844998744 

Calculating statistics on variogram model fit...
  Variogram model parameters used by PyKrige: [8.82533682e+00 1.11252177e+04 6.06592784e+00]
Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Hourly surfaces:   1%|▎                                  | 1/96 [00:00<00:33,  2.87it/s]

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

--- End Kriging Debug ---

[2025-03-30 00:00Z] → Variable: temp_wet, Source: station, points=40, lapse_apply=-0.00366
    min=-7.320, max=8.702, mean=-0.276, var=13.730947

--- Kriging Debug ---
  n points = 40
  value min/max = 15.000/79.000, mean=36.708, var=190.563301
  variogram_model = spherical
  variogram_params input = None
  Falling back to auto-fit (variogram_params invalid or None).
Adjusting data for anisotropy...
Initializing variogram model...
Coordinates type: 'euclidean' 

Using 'spherical' Variogram Model
Partial Sill: 71.61621658057093
Full Sill: 195.1120685193357
Range: 78313.77491399879
Nugget: 123.49585193876477 

Calculating stat

Hourly surfaces:   2%|▋                                  | 2/96 [00:00<00:32,  2.86it/s]

...

Hourly surfaces:  99%|█████████████████████████████████▋| 95/96 [01:39<00:01,  1.59s/it]

Executing Ordinary Kriging...

Executing Ordinary Kriging...

--- End Kriging Debug ---

[2025-04-02 22:00Z] → Variable: plp, Source: imerg, points=330, lapse_apply=0.00000
    min=0.000, max=100.000, mean=37.788, var=1852.149397
[2025-04-02 23:00Z] dynamic lapse = -0.0051 °C/m

--- Kriging Debug ---
  n points = 40
  value min/max = -4.790/19.059, mean=2.702, var=20.503752
  variogram_model = spherical
  variogram_params input = None
  Falling back to auto-fit (variogram_params invalid or None).
Adjusting data for anisotropy...
Initializing variogram model...
Coordinates type: 'euclidean' 

Using 'spherical' Variogram Model
Partial Sill: 10.184746265338884
Full Sill: 15.928203652722562
Range: 2907.743499392543
Nugget: 5.743457387383679 

Calculating statistics on variogram model fit...
  Variogram model parameters used by PyKrige: [  10.18474627 2907.74349939    5.74345739]
Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary 

Hourly surfaces: 100%|██████████████████████████████████| 96/96 [01:40<00:00,  1.54s/it]

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

Executing Ordinary Kriging...

--- End Kriging Debug ---

[2025-04-02 23:00Z] → Variable: plp, Source: imerg, points=330, lapse_apply=0.00000
    min=0.000, max=100.000, mean=37.788, var=1852.149397


Hourly surfaces: 100%|██████████████████████████████████| 96/96 [01:40<00:00,  1.05s/it]


In [24]:
# ============================= 7) SAVE =============================

ds = xr.Dataset(
    {**{k: xr.DataArray(v, coords=coords, dims=("time","y","x")) for k, v in data_vars.items()},
     "elev": xr.DataArray(dem_data.astype(np.float32), coords={"y": y_centers, "x": x_centers}, dims=("y","x"))},
    attrs={
        "title": "Hourly predictor stacks on 1-km grid (Ordinary Kriging)",
        "interpolation_method": "Ordinary Kriging",
        "variogram_model": CONFIG["variogram_model"],
        "variogram_strategy": CONFIG["variogram_strategy"],
        "test_window": f"{CONFIG['test_start']} → {CONFIG['test_end']}",
    }
)

# CF mapping and CRS
ds = ds.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
ds = ds.rio.write_crs(dem_profile["crs"], grid_mapping_name="spatial_ref")
ds = ds.rio.write_transform(dem_profile["transform"])
for v in ds.data_vars:
    ds[v].attrs.setdefault("grid_mapping", "spatial_ref")
A = dem_profile["transform"]
ds.attrs["GeoTransform"] = f"{A.c} {A.a} {A.b} {A.f} {A.d} {A.e}"

# Ensure time is tz-naive before NetCDF
if hasattr(ds.indexes.get("time", None), "tz") and ds.indexes["time"].tz is not None:
    ds = ds.assign_coords(time=ds.indexes["time"].tz_localize(None))

# Write compressed NetCDF
out_nc = OUT_DIR / "hourly_predictors_1km_kriging_v3_test.nc"

def _chunks_for(da):
    if da.dims == ("time", "y", "x"):
        return (min(24, da.sizes["time"]), min(256, da.sizes["y"]), min(256, da.sizes["x"]))
    if da.dims == ("y", "x"):
        return (min(256, da.sizes["y"]), min(256, da.sizes["x"]))
    return None

encoding = {}
for name, da in ds.data_vars.items():
    ch = _chunks_for(da)
    encoding[name] = ({"zlib": True, "complevel": 4, "chunksizes": ch}
                      if ch is not None else {"zlib": True, "complevel": 4} if da.ndim > 0 else {})

ds.to_netcdf(out_nc, engine="netcdf4", encoding=encoding)
print(f"Wrote {out_nc} (compressed).")


Wrote C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\hourly_predictors_1km_kriging_v3_test.nc (compressed).


In [25]:
# -------------------- Quick Plotting ------------------------------------

from pyproj import CRS

def quicklook_hour(
    ds, t, st_t, mros_t, out_png,
    vars_to_show=("plp","mros_plp_proxy","temp_air","temp_dew","temp_wet","rh")
):
    # match time index
    times_ds = pd.to_datetime(ds.time.values).floor("h")
    t_floor  = pd.to_datetime(t).floor("h")
    if t_floor not in times_ds.values:
        print(f"No matching time {t_floor} in dataset for quicklook.")
        return
    ti = int(np.where(times_ds == t_floor)[0][0])

    # axes extent (xmin, xmax, ymin, ymax)
    xvals = ds["x"].values
    yvals = ds["y"].values
    xmin, xmax = float(np.min(xvals)), float(np.max(xvals))
    ymin, ymax = float(np.min(yvals)), float(np.max(yvals))
    extent = [xmin, xmax, ymin, ymax]

    keep = [v for v in vars_to_show if v in ds.data_vars]
    if not keep:
        print("No matching variables to plot.")
        return
    ncols, nrows = 3, int(np.ceil(len(keep)/3))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 3.8*nrows), squeeze=False)
    fig.suptitle(f"Quicklook @ {t_floor:%Y-%m-%d %H:%MZ}", fontsize=14)

    # dataset CRS (fallback to configured)
    target_crs = ds.rio.crs or CRS.from_user_input(CONFIG["proj_fallback"])
    tf = Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)

    # --- project & CLIP stations ---
    st_x = np.empty(0)
    st_y = np.empty(0)
    if len(st_t):
        sx, sy = tf.transform(st_t["lon"].values, st_t["lat"].values)
        sx = np.asarray(sx); sy = np.asarray(sy)
        smask = (sx >= xmin) & (sx <= xmax) & (sy >= ymin) & (sy <= ymax) & np.isfinite(sx) & np.isfinite(sy)
        st_x, st_y = sx[smask], sy[smask]

    # --- project & CLIP MRoS ---
    mo_x = np.empty(0)
    mo_y = np.empty(0)
    if len(mros_t):
        mx, my = tf.transform(mros_t["lon"].values, mros_t["lat"].values)
        mx = np.asarray(mx); my = np.asarray(my)
        mmask = (mx >= xmin) & (mx <= xmax) & (my >= ymin) & (my <= ymax) & np.isfinite(mx) & np.isfinite(my)
        mo_x, mo_y = mx[mmask], my[mmask]

    for i, var in enumerate(keep):
        ax = axes[i // ncols, i % ncols]
        arr = ds[var].isel(time=ti).values

        # color scaling
        if var in ("plp", "mros_plp_proxy", "rh"):
            im = ax.imshow(arr, origin="upper", extent=extent, aspect="equal", vmin=0, vmax=100)
        else:
            im = ax.imshow(arr, origin="upper", extent=extent, aspect="equal")

        ax.set_title(var)
        ax.set_xlabel("Easting (km)")
        ax.set_ylabel("Northing (km)")
        ax.ticklabel_format(style="plain")   # disable 1e6 scientific format
        ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
        ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])

        # overlay
        if st_x.size:
            ax.scatter(st_x, st_y, s=15, c="white", edgecolor="k",
                    marker="o", linewidths=0.5, label="Stations", zorder=3)
        if mo_x.size:
            ax.scatter(mo_x, mo_y, s=25, c="red", edgecolor="k",
                    marker="^", linewidths=0.6, label="MRoS", zorder=3)

        ax.legend(loc="upper right", frameon=True, fontsize=8)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

    # turn off any leftover panels
    for j in range(len(keep), nrows*ncols):
        axes[j // ncols, j % ncols].axis("off")

    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    fig.savefig(out_png, dpi=200)
    plt.close(fig)
    print(
        f"Saved quicklook: {out_png} | plotted {st_x.size} stations, {mo_x.size} MRoS (clipped to DEM)"
    )


# -------------------- Loop --------------------

quick_dir = Path(CONFIG["out_dir"]) / "maps"
quick_dir.mkdir(parents=True, exist_ok=True)

# day = "2025-03-04"
# all_times = pd.to_datetime(ds.time.values).floor("h")  # dataset times
# mask = all_times.normalize() == pd.to_datetime(day)
# sample_hours = all_times[mask]
sample_hours = pd.to_datetime(ds.time.values)[::max(1, len(ds.time)//20)]
# print(f"Found {len(sample_hours)} timesteps on {day}")

for t in sample_hours:
    t_floor = pd.to_datetime(t).floor("h")  # tz-naive

    # Ensure obs times are made tz-naive before comparison
    st_t   = st_hr[st_hr["hour_utc"].dt.tz_convert(None).dt.floor("h") == t_floor]
    mros_t = mros_hr[mros_hr["hour_utc"].dt.tz_convert(None).dt.floor("h") == t_floor]

    print(f"[{t_floor}] Stations: {len(st_t)}, MRoS: {len(mros_t)}")

    quicklook_hour(ds, t_floor, st_t, mros_t,
                   out_png=quick_dir / f"test3_kriging_quick_{print_time(t_floor).replace(':','-')}.png")


[2025-03-30 00:00:00] Stations: 40, MRoS: 0


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test3_kriging_quick_2025-03-30 00-00Z.png | plotted 40 stations, 0 MRoS (clipped to DEM)
[2025-03-30 04:00:00] Stations: 39, MRoS: 0


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test3_kriging_quick_2025-03-30 04-00Z.png | plotted 39 stations, 0 MRoS (clipped to DEM)
[2025-03-30 08:00:00] Stations: 40, MRoS: 0


...

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test3_kriging_quick_2025-04-02 12-00Z.png | plotted 40 stations, 1 MRoS (clipped to DEM)
[2025-04-02 16:00:00] Stations: 40, MRoS: 0


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test3_kriging_quick_2025-04-02 16-00Z.png | plotted 40 stations, 0 MRoS (clipped to DEM)
[2025-04-02 20:00:00] Stations: 40, MRoS: 5


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test3_kriging_quick_2025-04-02 20-00Z.png | plotted 40 stations, 5 MRoS (clipped to DEM)
